<a href="https://colab.research.google.com/github/Eriyon24/Blind-date-Detroit-project/blob/main/Blind_Date_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


### Upload survey file

In [ ]:
uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}"')
  survey_results = pd.read_csv(filename)
  print(f'DataFrame df1 loaded from {filename}. First 5 rows:')
  display(survey_results.head())

### Upload round2 signups

In [ ]:
uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}"')
  round2_signups = pd.read_csv(filename)
  print(f'DataFrame df2 loaded from {filename}. First 5 rows:')
  display(round2_signups.head())

### Upload round 1 matches lookp

In [ ]:
uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}"')
  round_one_results = pd.read_csv(filename)
  print(f'DataFrame df3 loaded from {filename}. First 5 rows:')
  display(round_one_results.head())

Upload

In [ ]:
#Cleaning cities
df["city_clean"] = (
    df["City/Area"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["city_clean"] = df["city_clean"].replace({
     "st. clair shores": "saint clair shores",
    "st clair shores": "saint clair shores",
    "clinton twp": "clinton township",
    "canton/plymouth": "canton",
    "ann arbor/ypsilanti": "ann arbor",
    "royal oak, mi": "royal oak",
    "detroit, mi": "detroit",
    "brighton/detroit": "brighton"
})

In [ ]:
#Clean column names
df.columns = ([
    "name",
    "email",
    "age",
    "gender",
    "city_area",
    "interested_in",
    "relationship_goal",
    "preferred_age_ranges",
    "interests",
    "ideal_first_date",
    "Short Bio/About You",
    "best_days_available",
    "best_time_availability"
])

print(df.columns.tolist())



#Rename columns
df = df.rename(columns={
    "best_time_availabilty": "best_time_availabilty",
    "preferred_age_range(s)": "preferred_age_range"
})

df.columns

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(df.columns.tolist())

In [ ]:
df = df.rename(columns={
    "preferred_age_range(s)": "preferred_age_range",
})

print(df.columns.tolist())

In [ ]:
# Split age range
import re

def get_min_age(age_range_text):
    numbers = re.findall(r"\d+", str(age_range_text))
    numbers = [int(num) for num in numbers]
    return min(numbers)

def get_max_age(age_range_text):
    numbers = re.findall(r"\d+", str(age_range_text))
    numbers = [int(num) for num in numbers]
    return max(numbers)

df["min_age"] = df["preferred_age_range"].apply(get_min_age)
df["max_age"] = df["preferred_age_range"].apply(get_max_age)

df[["name", "preferred_age_range", "min_age", "max_age"]].head()

In [ ]:
# Combine interests + bio

df["combined_text"] = (
    df["interests"].fillna("") + " " +
    df["short_bio/about_you"].fillna("") + " " +
    df["ideal_first_date"].fillna("")
)

df[["name", "combined_text"]].head()

In [ ]:
#NLP turn text into similarity scores

vectorizer = TfidfVectorizer(
    stop_words="english"
)

matrix = vectorizer.fit_transform(
    df["combined_text"]
)

similarity = cosine_similarity(matrix)

print(similarity)

In [ ]:
def older_participant_bonus(person1, person2):

    #Both people are 50+
    if person1["age"] >= 50 and person2["age"] >= 50:
        return 200

    #Both people are 40+
    if person1["age"] >= 40 and person2["age"] >= 40:
        return 150

    #Both people are 36+
    if person1["age"] >= 36 and person2["age"] >= 36:
        return 100

    return 0

In [ ]:
def get_county(city):

    oakland = [
        "royal oak", "ferndale", "troy", "southfield",
        "novi", "farmington hills", "holly", "waterford",
        "auburn hills", "clarkston", "franklin","rochester hills","birmingham","west bloomfield","northville"
    ]

    wayne = [
        "detroit", "dearborn", "livonia", "westland",
        "canton", "plymouth","grosse pointe"
    ]

    macomb = [
        "sterling heights", "clinton township",
        "roseville", "saint clair shores", "macomb township"
    ]

    washtenaw = [
        "ann arbor", "ypsilanti","brighton","south lyon"
    ]

    if city in oakland:
        return "oakland"

    if city in wayne:
        return "wayne"

    if city in macomb:
        return "macomb"

    if city in washtenaw:
        return "washtenaw"

    return "unknown"

In [ ]:
# West / far city lock rule

west_lock_cities = [
    "ann arbor",
    "ypsilanti",
    "brighton",
    "south lyon",
    "northville",
    "canton",
    "plymouth",
    "west bloomfield",
    "farmington hills"
]

def west_city_lock(person1, person2):

    city1 = person["city_clean"]
    city1 = person2["city_clean"]

    if city1 in west_lock_cities and city2 not in west_lock_cities:
        return False

    if city2 in west_lock_cities and city1 not in west_lock_cities:
        return false

    return True

In [ ]:
def calculate_match_score(person1, person2, i, j):

    score = 0

    # your gender checks here

    # your age checks here

    # west city lock
    if not west_city_lock(person1, person2):
        return 0

    # rest of your scoring below

In [ ]:
#Create county column

df["county_clean"] = df["city_clean"].apply(get_county)

In [ ]:
#location bonus function
def location_bonus(person1, person2):

    county1 = person1["county_clean"]
    county2 = person2["county_clean"]

    # same county
    if county1 == county2:
        return 15

    nearby_pairs = [
        ("oakland", "wayne"),
        ("oakland", "macomb"),
        ("wayne", "macomb"),
        ("wayne", "washtenaw"),
        ("oakland", "washtenaw")
    ]

    pair = tuple(sorted([county1, county2]))

    nearby_pairs_clean = [
        tuple(sorted(p)) for p in nearby_pairs
    ]

    # nearby county
    if pair in nearby_pairs_clean:
        return 8

    return 0

In [ ]:
#Bio key words for matching
bio_keywords = {
    "fitness": ["gym", "fitness", "working out", "active", "training"],
    "food": ["food", "restaurants", "coffee", "brunch", "dinner"],
    "travel": ["travel", "adventure", "explore"],
    "family": ["family", "kids", "children"],
    "faith": ["christian", "faith", "church", "god"],
    "outdoors": ["hiking", "nature", "outdoors", "walks"],
    "creative": ["art", "music", "creative", "writing"],
    "career": ["career", "ambitious", "driven", "business"]
}

In [ ]:
#Function to find bio traits
def get_bio_traits(bio):

    bio = str(bio).lower()
    traits = []

    for trait, words in bio_keywords.items():
        for word in words:
            if word in bio:
                traits.append(trait)
                break

    return set(traits)

In [ ]:
#Bio boosting for matching
def bio_bonus(person1, person2):

    traits1 = get_bio_traits(person1["short_bio/about_you"])
    traits2 = get_bio_traits(person2["short_bio/about_you"])

    shared_traits = traits1.intersection(traits2)

    return len(shared_traits) * 10

In [ ]:
# Matching Function
def calculate_match_score(person1, person2, i, j):

    score = 0

    score += location_bonus(person1, person2)

    score += older_participant_bonus(person1,person2)

    score += bio_bonus(person1, person2)


    #36+ bonus

    score += age_36_plus_bonus(
        person1,
        person2
    )

    #Gender
    if person1["gender"] != person2["interested_in"]:
        return 0

    if person2["gender"] != person1["interested_in"]:
        return 0


    #Older participants 36+

    age_difference = abs(
        person1["age"] - person2["age"]
    )




    if person1["age"] >= 36 or person2["age"] >= 36:

        if age_difference > 15:
            return 0

    #Normal matching for younger particpants

    #Age
    if not(
        person1["min_age"]
        <= person2["age"]
        <= person1["max_age"]
    ):
        return 0

    if not (
        person2["min_age"]
        <= person1["age"]
        <= person2["max_age"]
    ):
        return 0

    # Relationship Goal
    if (
        person1["relationship_goal"]
        ==
        person2["relationship_goal"]
    ):
        score += 20


    elif (
        person1["relationship_goal"] == "Open to Either"
        or
        person2["relationship_goal"] == "Open to Either"
    ):

        score += 15


           # Shared Interests
    interests1 = set(
        str(person1["interests"]).lower().split(",")
    )

    interests2 = set(
        str(person2["interests"]).lower().split(",")
    )

    shared_interests = interests1.intersection(interests2)

    score += len(shared_interests) * 10


    # Ideal First Date Match
    if (
        person1["ideal_first_date"]

        ==
        person2["ideal_first_date"]
    ):

        score += 20

    # NLP BIO SCORE
    score += similarity[i][j] * 50

    return score

In [ ]:
#Replace gender labels to match
df["gender"] = df["gender"].replace({
    "Male" : "Men",
    "Female": "Women"
})

print(df["gender"].unique())
print(df["interested_in"].unique())

In [ ]:
print(df["gender"].unique())
print(df["interested_in"].unique())
print(df["relationship_goal"].unique())

In [ ]:
#fix age columns
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["min_age"] = pd.to_numeric(df["min_age"], errors="coerce")
df["max_age"] = pd.to_numeric(df["max_age"], errors="coerce")

print(df[["age", "min_age", "max_age"]].dtypes)

In [ ]:
def age_36_plus_bonus(person1, person2):

    if person1["age"] >= 36 and person2["age"] >= 36:
        return 100

    return 0

In [ ]:
# Find Matches

matches = []

MAX_MATCHES = 4

match_counts = {}

max_times_matched = 2

for i in range(len(df)):

    person1 = df.iloc[i]
    person1_name = person1["name"]

    best_match = None

    best_score = 0

    for j in range(len(df)):

        #Skip same person
        if i == j:
            continue

        person2 = df.iloc[j]

        person2_name = person2["name"]

        #skip this person if they already have too many matches
        if match_counts.get(person2_name, 0) >= max_times_matched:
            continue


        # Calculate score
        score = calculate_match_score(
            person1,
            person2,
            i,
            j
        )

        # Save best match
        if score > best_score:

            best_score = score

            best_match = person2["name"]

        #Find best match email:
        if best_match is not None:

            best_match_email = df[
                df["name"] == best_match
            ]["email"].values[0]

        else:

            best_match_email = None





        # skip if already at limit
        if match_counts.get(person1["name"], 0) >= MAX_MATCHES:
            continue

        if match_counts.get(person2["name"], 0) >= MAX_MATCHES:
            continue


           # Save result
    matches.append({

        "Person": person1["name"],

        "Best Match": best_match,

        "Compatibility Score": best_score,

        "Person Email": person1["email"],

        "Best Match Email": best_match_email
    })


        # increase match counts
    match_counts[person1["name"]] = (
        match_counts.get(person1["name"], 0) + 1
    )

    match_counts[person2["name"]] = (
        match_counts.get(person2["name"], 0) + 1
    )

        # count how many time someone got matched
    if best_match is not None and best_score > 0:
        match_counts[person1_name] = match_counts.get(person1_name, 0) + 1
        match_counts[best_match] = match_counts.get(best_match, 0) + 1



results_df = pd.DataFrame(matches)

results_df

In [ ]:
#no match results
no_match_df = results_df[
(results_df["Best Match"].isna()) |
(results_df["Compatibility Score"] == 0)
 ]

no_match_df

In [ ]:
#second pass matches
second_pass_matches = []

for i in range(len(no_match_df)):

    person_name = no_match_df.iloc[i]["Person"]

    person1 = df[df["name"] == person_name].iloc[0]

    person1_index = df[df["name"] == person_name].index[0]

    best_score = 0
    best_match = None

    for j in range(len(df)):

        person2 = df.iloc[j]

        if person1["name"] == person2["name"]:
            continue

        score = calculate_match_score(
            person1,
            person2,
            person1_index,
            j
        )

        if score > best_score:
            best_score = score
            best_match = person2["name"]

    second_pass_matches.append({
        "Person": person1["name"],
        "Second Pass Match": best_match,
        "Compatibility Score": best_score
    })

second_pass_df = pd.DataFrame(second_pass_matches)

second_pass_df

In [ ]:
# Get names of people who did not match
no_match_names = no_match_df["Person"].tolist()

#Go back to orginal df and find unmatched people age 36+
older_no_match_df = df[
    (df["name"].isin(no_match_names)) &
    (df["age"] >= 36)
]

older_no_match_df

In [ ]:
print(older_no_match_df[["name", "age", "gender", "interested_in"]])
print(len(older_no_match_df))

In [ ]:
# Redo calculate older matches
def calculate_older_match_score(person1, person2):

    score = 0

    # skip same person
    if person1["name"] == person2["name"]:
        return 0

    # both must be 36+
    if person1["age"] < 36 or person2["age"] < 36:
        return 0

    # gender preference check
    if person1["interested_in"] != person2["gender"]:
        return 0

    if person2["interested_in"] != person1["gender"]:
        return 0

    score += 100

    return score

In [ ]:
#Test
person1 = older_no_match_df.iloc[0]
person2 = older_no_match_df.iloc[1]

print(calculate_older_match_score(person1, person2))

In [ ]:
# match older people with all older people
older_pool = df[df["age"] >= 36]

older_matches = []

for i in range(len(older_no_match_df)):

    person1 = older_no_match_df.iloc[i]

    best_score = 0
    best_match = None

    for j in range(len(older_pool)):

        person2 = older_pool.iloc[j]

        if person1["name"] == person2["name"]:
            continue

        score = calculate_older_match_score(person1, person2)

        if score > best_score:
            best_score = score
            best_match = person2["name"]

    older_matches.append({
        "Person": person1["name"],
        "Older Match": best_match,
        "Compatibility Score": best_score

    })

older_matches_df = pd.DataFrame(older_matches)

#Limit matches per person
older_matches_df["match_rank"] = (
    older_matches_df.groupby("Older Match")
    .cumcount() + 1
)

older_matches_df = older_matches_df[
    older_matches_df["match_rank"] <= 3
]

older_matches_df = older_matches_df.drop(
    columns=["match_rank"]
)

older_matches_df.head(10)

In [ ]:
# Force Eriyon to get a match

my_name = "Eriyon Adams"

my_index = df[df["name"] == my_name].index[0]
me = df.loc[my_index]

my_matches = []

for j in range(len(df)):

    other_person = df.iloc[j]

    if other_person["name"] == my_name:
        continue

    score = calculate_match_score(
        me,
        other_person,
        my_index,
        j
    )

    if score > 0:


      my_matches.append({
            "Person": my_name,
            "Match": other_person["name"],
            "Match Email": other_person["email"],
            "Score": score
        })

my_matches_df = pd.DataFrame(my_matches)

my_matches_df = my_matches_df.sort_values(
    by="Score",
    ascending=False
)

my_top_matches = my_matches_df.head(4)

my_top_matches

In [ ]:
#email

import smtplib

from email.message import EmailMessage

sender_email = "blindmatchdetroit@gmail.com"

app_password = "ylad sdrj tsuz zdim"

In [ ]:
def first_name(name):
    return str(name).strip().split()[0]

In [ ]:
# Clean column names for main table
main_matches_clean = results_df.rename(columns={
    "Best Match": "Match",
    "Best Match Email": "Match Email",
    "Compatibility Score": "Score"
})[
    ["Person", "Person Email", "Match", "Match Email", "Score",]
]

# Lookup table from orginal df
info_lookup = df.drop_duplicates(subset="name").set_index("name")


# Add email columns
main_matches_clean["Person Email"] = main_matches_clean["Person"].map(
    info_lookup["email"]
)


main_matches_clean["Match Email"] = main_matches_clean["Match"].map(
    info_lookup["email"]
)

# Add match details from original df
main_matches_clean["ideal_first_date"] = main_matches_clean["Match"].map(
    info_lookup["ideal_first_date"]
)

main_matches_clean["short_bio/about_you"] = main_matches_clean["Match"].map(
    info_lookup["short_bio/about_you"]
)

main_matches_clean["best_days_available"] = main_matches_clean["Match"].map(
    info_lookup["best_days_available"]
)

main_matches_clean["best_time_availabilty"] = main_matches_clean["Match"].map(
    info_lookup["best_time_availabilty"]
)



# NOW select final columns
main_matches_clean = main_matches_clean[
    [
        "Person",
        "Person Email",
        "Match",
        "Match Email",
        "Score",
        "ideal_first_date",
        "short_bio/about_you",
        "best_days_available",
        "best_time_availabilty"
    ]
]

main_matches_clean.head()

In [ ]:
# Clean columns for old matches

older_matches_clean = older_matches_df.rename(columns={
    "Older Match": "Match",
    "Compatibility Score": "Score"
})

# Lookup table from original df
info_lookup = df.drop_duplicates(subset="name").set_index("name")

# Add email columns
older_matches_clean["Person Email"] = older_matches_clean["Person"].map(
    info_lookup["email"]
)

older_matches_clean["Match Email"] = older_matches_clean["Match"].map(
    info_lookup["email"]
)

# Add match details from original df
older_matches_clean["ideal_first_date"] = older_matches_clean["Match"].map(
    info_lookup["ideal_first_date"]
)

older_matches_clean["short_bio/about_you"] = older_matches_clean["Match"].map(
    info_lookup["short_bio/about_you"]
)

older_matches_clean["best_days_available"] = older_matches_clean["Match"].map(
    info_lookup["best_days_available"]
)

older_matches_clean["best_time_availabilty"] = older_matches_clean["Match"].map(
    info_lookup["best_time_availabilty"]
)

# NOW select final columns
older_matches_clean = older_matches_clean[
    [
        "Person",
        "Person Email",
        "Match",
        "Match Email",
        "Score",
        "ideal_first_date",
        "short_bio/about_you",
        "best_days_available",
        "best_time_availabilty"
    ]
]

older_matches_clean.head()

In [ ]:
# Clean columns for older matches

second_matches_clean = second_pass_df.rename(columns={
    "Second Pass Match": "Match",
    "Compatibility Score": "Score"
})

# Lookup table from original df
info_lookup = df.drop_duplicates(subset="name").set_index("name")

# Add email columns
second_matches_clean["Person Email"] = second_matches_clean["Person"].map(
    info_lookup["email"]
)

second_matches_clean["Match Email"] = second_matches_clean["Match"].map(
    info_lookup["email"]
)

# Add match details from original df
second_matches_clean["ideal_first_date"] = second_matches_clean["Match"].map(
    info_lookup["ideal_first_date"]
)

second_matches_clean["short_bio/about_you"] = second_matches_clean["Match"].map(
    info_lookup["short_bio/about_you"]
)

second_matches_clean["best_days_available"] = second_matches_clean["Match"].map(
    info_lookup["best_days_available"]
)

second_matches_clean["best_time_availabilty"] = second_matches_clean["Match"].map(
    info_lookup["best_time_availabilty"]
)

# NOW select final columns
second_matches_clean = second_matches_clean[
    [
        "Person",
        "Person Email",
        "Match",
        "Match Email",
        "Score",
        "ideal_first_date",
        "short_bio/about_you",
        "best_days_available",
        "best_time_availabilty"
    ]
]

second_matches_clean.head()

In [ ]:
print(second_pass_df.columns)

In [ ]:
all_matches_clean = pd.concat([
    main_matches_clean,
    my_top_matches,
    second_matches_clean,
    older_matches_clean
], ignore_index=True)

all_matches_clean.head().reset_index()

info_lookup = df.drop_duplicates(subset="name").set_index("name")

all_matches_clean["Person Ideal First Date"] = all_matches_clean["Person"].map(
    info_lookup["ideal_first_date"]
)

all_matches_clean["Person Bio"] = all_matches_clean["Person"].map(
    info_lookup["short_bio/about_you"]
)

all_matches_clean["Match Ideal First Date"] = all_matches_clean["Match"].map(
    info_lookup["ideal_first_date"]
)

all_matches_clean["Match Bio"] = all_matches_clean["Match"].map(
    info_lookup["short_bio/about_you"]
)

all_matches_clean["Person Age"] = all_matches_clean["Person"].map(
    info_lookup["age"]
)

all_matches_clean["Match Age"] = all_matches_clean["Match"].map(
    info_lookup["age"]
)

all_matches_clean["Match best_days_available"] = all_matches_clean["Match"].map(
    info_lookup["best_days_available"]
)

all_matches_clean["Person best_days_available"] = all_matches_clean["Person"].map(
    info_lookup["best_days_available"]
)

all_matches_clean["Match best_time_availabilty"] = all_matches_clean["Match"].map(
    info_lookup["best_time_availabilty"]
)

all_matches_clean["Person best_time_availabilty"] = all_matches_clean["Person"].map(
    info_lookup["best_time_availabilty"]

)

all_matches_clean["Person city/area"] = all_matches_clean["Person"].map(
    info_lookup["city/area"]
)

all_matches_clean["Match city/area"] = all_matches_clean["Match"].map(
    info_lookup["city/area"]
)

# NOW reorder columns
all_matches_clean = all_matches_clean[
    [
        "Person",
        "Person Email",
        "Person Ideal First Date",
        "Person Bio",
        "Person Age",
        "Person best_days_available",
        "Person best_time_availabilty",
        "Person city/area",

        "Match",
        "Match Email",
        "Match Ideal First Date",
        "Match Bio",
        "Match Age",
        "Match best_days_available",
        "Match best_time_availabilty",
        "Match city/area",



        "Score"
    ]
]

# preview
all_matches_clean.head()

In [ ]:
all_matches_clean.columns.tolist()

In [ ]:
print(len(all_matches_clean))

In [ ]:
# Remove rows where Match is missing
all_matches_clean = all_matches_clean.dropna(
    subset=["Match"]
)

# Remove rows where Match says No Match
all_matches_clean = all_matches_clean[
    all_matches_clean["Match"] != "No Match"
]

# Remove exact duplicate rows
all_matches_clean = all_matches_clean.drop_duplicates()

# Create duplicate-pair key
all_matches_clean["pair_key"] = all_matches_clean.apply(
    lambda row: tuple(sorted([
        str(row["Person"]),
        str(row["Match"])
    ])),
    axis=1
)

# Remove duplicate pairs like A-B and B-A
all_matches_clean = all_matches_clean.drop_duplicates(
    subset=["pair_key"]
)

# Drop helper column
all_matches_clean = all_matches_clean.drop(
    columns=["pair_key"]
)

# FINAL CLEAN TABLE
all_matches_clean = all_matches_clean.drop_duplicates()

# -------------------------
# ADD THIS RIGHT HERE
# -------------------------

# Full table (everyone)
all_matches_full = all_matches_clean.copy()

# Real matches only
match_emails_df = all_matches_full[
    all_matches_full["Match"].notna()
]

match_emails_df = match_emails_df[
    match_emails_df["Match"] != "No Match"
]

# No match people
no_match_df = all_matches_full[
    all_matches_full["Match"].isna()
]

# Preview counts
print("All people:", len(all_matches_full))
print("Matched people:", len(match_emails_df))
print("No match people:", len(no_match_df))

In [ ]:
all_matches_clean.to_csv(r"\\ad.wayne.edu\Users\ex4360\Documents\Blind Date Detroit\Match Rounds\all_matches_clean.csv")

In [ ]:
# Find row index to change ma
pd.set_option("display.max_rows", None)

match_emails_df.reset_index()[[
    "index",
    "Person",
    "Match"
]]

In [ ]:
#Change Eriyon's bio
match_emails_df.loc[
    match_emails_df["Person"] == "Eriyon Adams",
    "Person Bio"
] = """
Easygoing and driven guy who enjoys staying active, trying new things, and meeting new people. Into fitness, good conversations, social events, and keeping life fun and low pressure.
"""

In [ ]:
# change Eriyon match
match_emails_df.at[81, "Match"] = "girl"

# lookup
info_lookup = df.drop_duplicates(subset="name").set_index("name")

# update match details
match_emails_df.at[81, "Match Email"] = info_lookup.at["girl", "email"]
match_emails_df.at[81, "Match Bio"] = info_lookup.at["girl", "short_bio/about_you"]
match_emails_df.at[81, "Match Age"] = info_lookup.at["girl", "age"]
match_emails_df.at[81, "Match Ideal First Date"] = info_lookup.at["girl", "ideal_first_date"]
match_emails_df.at[81, "Match city/area"] = info_lookup.at["girl", "city/area"]

In [ ]:
match_emails_df.loc[33]

In [ ]:
#Manual adding people
new_match = match_emails_df.iloc[0].copy()

new_match["Person"] = "person"
new_match["Match"] = "Alex"

info_lookup = df.set_index("name")

# person info
new_match["Person Email"] = info_lookup.at[
    "person",
    "email"
]

# match info
new_match["Match Email"] = info_lookup.at[
    "new",
    "email"
]

new_match["Match Bio"] = info_lookup.at[
    "new",
    "short_bio/about_you"
]

new_match["Match Age"] = info_lookup.at[
    "new",
    "age"
]


match_emails_df = pd.concat(
[match_emails_df, pd.DataFrame([new_match])],
ignore_index=True
)

In [ ]:
#Check to see if match is added
match_emails_df[
    match_emails_df["Person"] == "new"
][[
    "Person",
    "Person Email",
    "Match",
    "Match Email"
]]

In [ ]:
# Second match for lansing girl

new_match2 = match_emails_df.iloc[0].copy()

# Find second guy
second_guy = df[
    df["name"].str.contains("new match", case=False, na=False)
].iloc[0]

# Rhiannon row
rhiannon_row = df[
    df["name"] == "lansing girl"
].iloc[0]

# Fill info
new_match2["Person"] = "lansing girl"
new_match2["Match"] = second_guy["name"]

new_match2["Person Email"] = lansinggirl_row["email"]
new_match2["Match Email"] = second_guy["email"]

new_match2["Match Bio"] = second_guy["short_bio/about_you"]
new_match2["Match Age"] = second_guy["age"]

# Add row
match_emails_df = pd.concat(
    [match_emails_df, pd.DataFrame([new_match2])],
    ignore_index=True
)

# Check
match_emails_df[
    match_emails_df["Person"] == "Lansing girl"
][["Person", "Match"]]

match_emails_df = match_emails_df.drop_duplicates(
    subset=["Person", "Match"]
)

In [ ]:
# Matching emails test

sent_pairs = set()

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:

    smtp.login(sender_email, app_password)

    def first_name(full_name):
        return str(full_name).split()[0]

    # clean names just in case
    df["name"] = df["name"].astype(str).str.strip()
    match_emails_df["Person"] = match_emails_df["Person"].astype(str).str.strip()
    match_emails_df["Match"] = match_emails_df["Match"].astype(str).str.strip()

    for index, row in match_emails_df.iterrows():

        person_name = row["Person"]
        match_name = row["Match"]
        score = row["Score"]

        person_first = first_name(person_name)
        match_first = first_name(match_name)

        if pd.isna(match_name) or score == 0:
            continue

        pair = tuple(sorted([person_name, match_name]))

        if pair in sent_pairs:
            continue

        sent_pairs.add(pair)

        # find people safely
        person_rows = df[df["name"] == person_name]
        match_rows = df[df["name"] == match_name]

        if person_rows.empty:
            print("Missing person in df:", person_name)
            continue

        if match_rows.empty:
            print("Missing match in df:", match_name)
            continue

        person = person_rows.iloc[0]
        match = match_rows.iloc[0]

        shared_days = set(str(person["best_days_available"]).split(",")).intersection(
            set(str(match["best_days_available"]).split(","))
        )

        shared_times = set(str(person["best_time_availabilty"]).split(",")).intersection(
            set(str(match["best_time_availabilty"]).split(","))
        )

        shared_days_text = ", ".join(shared_days)
        shared_times_text = ", ".join(shared_times)

        shared_days_output = (
            f"Shared Days: {shared_days_text}"
            if shared_days_text
            else "Shared Days: No overlapping days listed"
        )

        shared_times_output = (
            f"Shared Times: {shared_times_text}"
            if shared_times_text
            else "Shared Times: No overlapping times listed"
        )

        msg = EmailMessage()

        msg["Subject"] = "Your Blind Date Match"
        msg["From"] = sender_email
        msg["To"] = person["email"]
        msg["Cc"] = match["email"]

        email_body = f"""
Hey {person_first} and {match_first},

You two matched!

Shared Availability:
{shared_days_output}
{shared_times_output}

About {person_first}:

Ideal First Date:
{person["ideal_first_date"]}

Age: {person["age"]}
Area: {person["city/area"]}

Bio:
{person["short_bio/about_you"]}

About {match_first}:

Ideal First Date:
{match["ideal_first_date"]}

Age: {match["age"]}
Area: {match["city/area"]}

Bio:
{match["short_bio/about_you"]}

Feel free to reply to this email and connect.

Please be respectful and meet in public places.

- Blind Matchmaking Project
"""

        msg.set_content(email_body.strip())

        print(email_body)
        print("=" * 80)
        print(f"Previewed match email to {person_first} and {match_first}")

In [ ]:
# TEST MATCH EMAIL TO YOURSELF

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:

    smtp.login(sender_email, app_password)

    def first_name(full_name):
        return str(full_name).split()[0]

    # clean names
    df["name"] = df["name"].astype(str).str.strip()
    match_emails_df["Person"] = match_emails_df["Person"].astype(str).str.strip()
    match_emails_df["Match"] = match_emails_df["Match"].astype(str).str.strip()

    # grab first match row
    row = match_emails_df.iloc[0]

    person_name = row["Person"]
    match_name = row["Match"]
    score = row["Score"]

    person_first = first_name(person_name)
    match_first = first_name(match_name)

    person = df[df["name"] == person_name].iloc[0]
    match = df[df["name"] == match_name].iloc[0]

    # shared availability
    shared_days = set(
        str(person["best_days_available"]).split(",")
    ).intersection(
        set(str(match["best_days_available"]).split(","))
    )

    shared_times = set(
        str(person["best_time_availabilty"]).split(",")
    ).intersection(
        set(str(match["best_time_availabilty"]).split(","))
    )

    shared_days_text = ", ".join(shared_days)
    shared_times_text = ", ".join(shared_times)

    shared_days_output = (
        f"Shared Days: {shared_days_text}"
        if shared_days_text
        else "Shared Days: No overlapping days listed"
    )

    shared_times_output = (
        f"Shared Times: {shared_times_text}"
        if shared_times_text
        else "Shared Times: No overlapping times listed"
    )

    # create email
    msg = EmailMessage()

    msg["Subject"] = "TEST - Your Blind Date Match"
    msg["From"] = sender_email

    # SEND TO YOURSELF
    msg["To"] = "blindmatchdetroit@gmail.com"

    email_body = f"""
Hey {person_first} and {match_first},

You two matched!

Shared Availability:
{shared_days_output}
{shared_times_output}

About {person_first}:

Ideal First Date:
{person["ideal_first_date"]}

Age: {person["age"]}
Area: {person["city/area"]}

Bio:
{person["short_bio/about_you"]}

About {match_first}:

Ideal First Date:
{match["ideal_first_date"]}

Age: {match["age"]}
Area: {match["city/area"]}

Bio:
{match["short_bio/about_you"]}

Feel free to reply to this email and connect.

Please be respectful and meet in public places.

- Blind Matchmaking Project
"""

    msg.set_content(email_body.strip())

    smtp.send_message(msg)

    print("Test email sent successfully to yourself.")

In [ ]:
#test no match table
print(len(no_match_df))
no_match_df.head()

In [ ]:
# Rebulid
no_match_df = all_matches_full[
    (all_matches_full["Match"].isna()) |
    (all_matches_full["Match"] == "No Match")
]

print(len(no_match_df))
no_match_df.head()

In [ ]:
matched_people = set(match_emails_df["Person"])

no_match_df = df[
    ~df["name"].isin(matched_people)
]

no_match_df[[
    "name",
    "email"
]]

In [ ]:
# Find row index to change ma
pd.set_option("display.max_rows", None)

no_match_df.reset_index()[[
    "index",
    "name",
    "email"
]]

In [ ]:
print(len(no_match_df))

In [ ]:
# No match email preview

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:

    smtp.login(sender_email, app_password)

    # helper function
    def first_name(full_name):

        return str(full_name).split()[0]

    # clean names
    df["name"] = df["name"].astype(str).str.strip()

    for index, row in no_match_df.iterrows():

        person_name = row["name"]

        msg = EmailMessage()

        msg["Subject"] = "Blind Matchmaking Project Update"
        msg["From"] = sender_email
        msg["To"] = row["email"]

        email_body = f"""
Hey {first_name(person_name)},

Thank you again for participating in the Blind Matchmaking Project.

Unfortunately, I wasn’t able to find a compatible match for you this round. Please don’t feel discouraged — sometimes it simply comes down to participant ratios, preferences, and compatibility overlap.

I truly appreciate you participating and helping make this project possible.

- Eriyon
"""

        msg.set_content(email_body.strip())

        print(email_body)
        print("=" * 80)
        print(f"Previewed no-match email to {person_name}")

In [ ]:
#Save files
match_emails_df.to_csv(r"\\ad.wayne.edu\Users\ex4360\Documents\Blind Date Detroit\Match Rounds\match_emails_backup_before_sending.csv ", index=False)
no_match_df.to_csv(r"\\ad.wayne.edu\Users\ex4360\Documents\Blind Date Detroit\Match Rounds\no_match_backup_before_sending.csv", index=False)

In [ ]:
# TEST NO-MATCH EMAIL TO YOURSELF

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:

    smtp.login(sender_email, app_password)

    def first_name(full_name):
        return str(full_name).split()[0]

    # clean names
    df["name"] = df["name"].astype(str).str.strip()

    # grab first no-match row
    row = no_match_df.iloc[0]

    person_name = row["name"]

    msg = EmailMessage()

    msg["Subject"] = "TEST - Blind Matchmaking Project Update"
    msg["From"] = sender_email

    # SEND TO YOURSELF
    msg["To"] = "blindmatchdetroit@gmail.com"

    email_body = f"""
Hey {first_name(person_name)},

Thank you again for participating in the Blind Matchmaking Project.

Unfortunately, I wasn’t able to find a compatible match for you this round. Please don’t feel discouraged — sometimes it simply comes down to participant ratios, preferences, and compatibility overlap.

I truly appreciate you participating and helping make this project possible.

If feedback is positive overall, there will likely be another round in the future.

- Eriyon
"""

    msg.set_content(email_body.strip())

    smtp.send_message(msg)

    print("Test no-match email sent successfully.")

In [ ]:
# REAL MATCH EMAIL SEND

sent_pairs = set()

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:

    smtp.login(sender_email, app_password)

    def first_name(full_name):
        return str(full_name).split()[0]

    # clean names
    df["name"] = df["name"].astype(str).str.strip()
    match_emails_df["Person"] = match_emails_df["Person"].astype(str).str.strip()
    match_emails_df["Match"] = match_emails_df["Match"].astype(str).str.strip()

    for index, row in match_emails_df.iterrows():

        person_name = row["Person"]
        match_name = row["Match"]
        score = row["Score"]

        person_first = first_name(person_name)
        match_first = first_name(match_name)

        if pd.isna(match_name) or score == 0:
            continue

        pair = tuple(sorted([person_name, match_name]))

        if pair in sent_pairs:
            continue

        sent_pairs.add(pair)

        person_rows = df[df["name"] == person_name]
        match_rows = df[df["name"] == match_name]

        if person_rows.empty:
            print("Missing person:", person_name)
            continue

        if match_rows.empty:
            print("Missing match:", match_name)
            continue

        person = person_rows.iloc[0]
        match = match_rows.iloc[0]

        # shared availability
        shared_days = set(
            str(person["best_days_available"]).split(",")
        ).intersection(
            set(str(match["best_days_available"]).split(","))
        )

        shared_times = set(
            str(person["best_time_availabilty"]).split(",")
        ).intersection(
            set(str(match["best_time_availabilty"]).split(","))
        )

        shared_days_text = ", ".join(shared_days)
        shared_times_text = ", ".join(shared_times)

        shared_days_output = (
            f"Shared Days: {shared_days_text}"
            if shared_days_text
            else "Shared Days: No overlapping days listed"
        )

        shared_times_output = (
            f"Shared Times: {shared_times_text}"
            if shared_times_text
            else "Shared Times: No overlapping times listed"
        )

        # email
        msg = EmailMessage()

        msg["Subject"] = "Your Blind Match"
        msg["From"] = sender_email

        # REAL SEND
        msg["To"] = person["email"]
        msg["Cc"] = match["email"]

        email_body = f"""
Hey {person_first} and {match_first},

You two matched!

Shared Availability:
{shared_days_output}
{shared_times_output}

About {person_first}:

Ideal First Date:
{person["ideal_first_date"]}

Age: {person["age"]}
Area: {person["city/area"]}

Bio:
{person["short_bio/about_you"]}

About {match_first}:

Ideal First Date:
{match["ideal_first_date"]}

Age: {match["age"]}
Area: {match["city/area"]}

Bio:
{match["short_bio/about_you"]}

Feel free to reply to this email and connect.

Please keep things respectful, open-minded, and low pressure throughout the experience. Also remember to meet in public places and prioritize safety.

Thanks again for participating and helping make this project possible.

- Eriyon
"""

        msg.set_content(email_body.strip())

        smtp.send_message(msg)

        print(f"Sent match email to {person_first} and {match_first}")

In [ ]:
# REAL NO-MATCH EMAIL SEND

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:

    smtp.login(sender_email, app_password)

    def first_name(full_name):
        return str(full_name).split()[0]

    df["name"] = df["name"].astype(str).str.strip()

    for index, row in no_match_df.iterrows():

        person_name = row["name"]

        msg = EmailMessage()

        msg["Subject"] = "Blind Matchmaking Project Update"
        msg["From"] = sender_email

        # REAL SEND
        msg["To"] = row["email"]

        email_body = f"""
Hey {first_name(person_name)},

Thank you again for participating in the Blind Matchmaking Project.

Unfortunately, I wasn’t able to find a compatible match for you this round. Please don’t feel discouraged — sometimes it simply comes down to participant ratios, preferences, compatibility overlap, and overall location logistics.

I truly appreciate you participating and helping make this project possible.

If feedback is positive overall, there will likely be another round in the future.

- Eriyon
"""

        msg.set_content(email_body.strip())

        smtp.send_message(msg)

        print(f"Sent no-match email to {person_name}")

In [ ]:
import imaplib
import email
from email.header import decode_header

imap = imaplib.IMAP4_SSL("imap.gmail.com")
imap.login(sender_email, app_password)

imap.select('"[Gmail]/Sent Mail"')

status, messages = imap.search(
    None,
    '(SUBJECT "Your Blind Match")'
)

sent_ids = messages[0].split()

sent_recipients = set()

for msg_id in sent_ids:
    status, msg_data = imap.fetch(msg_id, "(RFC822)")
    msg = email.message_from_bytes(msg_data[0][1])

    to_field = msg.get("To", "")
    cc_field = msg.get("Cc", "")

    all_recipients = to_field + "," + cc_field

    for e in all_recipients.split(","):
        clean_email = e.strip().lower()
        if clean_email:
            sent_recipients.add(clean_email)

imap.logout()

sent_recipients

In [ ]:
match_emails_df["Person Email"] = match_emails_df["Person Email"].str.lower().str.strip()
match_emails_df["Match Email"] = match_emails_df["Match Email"].str.lower().str.strip()

In [ ]:
# CLEAN EMAILS FIRST

match_emails_df["Person Email"] = (
    match_emails_df["Person Email"]
    .astype(str)
    .str.lower()
    .str.strip()
)

match_emails_df["Match Email"] = (
    match_emails_df["Match Email"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# SHOULD RECEIVE EMAILS
# combines both Person Email + Match Email

should_receive_emails = set(
    match_emails_df["Person Email"]
).union(
    set(match_emails_df["Match Email"])
)

# CLEAN SENT EMAILS FROM GMAIL

sent_emails_clean = set(
    email.lower().strip()
    for email in sent_recipients
)

# FIND MISSING EMAILS

missing_emails = (
    should_receive_emails - sent_emails_clean
)

print(missing_emails)

# CREATE DATAFRAME OF MISSING PEOPLE

missing_match_emails_df = match_emails_df[
    (
        match_emails_df["Person Email"]
        .isin(missing_emails)
    )
    |
    (
        match_emails_df["Match Email"]
        .isin(missing_emails)
    )
]

# SHOW RESULTS

missing_match_emails_df[
    [
        "Person",
        "Person Email",
        "Match",
        "Match Email"
    ]
]